<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap2_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

chap 2 transformer



2.1.1 テキストのトークン化

Qwen を使ってトークナイザーがどのように文書を処理するのかを見てみる

In [25]:
from transformers import AutoTokenizer

# prompt = "It was a dark and stormy"
prompt = "It was a dark and stormy night. The"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B")
input_ids = tokenizer(prompt).input_ids
input_ids

[2132, 572, 264, 6319, 323, 13458, 88, 3729, 13, 576]

In [26]:
for t in input_ids:
  print(t, "\t:", tokenizer.decode(t))

2132 	: It
572 	:  was
264 	:  a
6319 	:  dark
323 	:  and
13458 	:  storm
88 	: y
3729 	:  night
13 	: .
576 	:  The


備考）トークナイザーの訓練はモデルの訓練とは明らかに異なる。モデルの訓練が確率的であるのに対し、トークナイザーの訓練は決定論的である。トークナイザーの訓練は与えられたトークンをどのように処理するのかを特定するためのもの。

2.1.2 確率の予測

GPT-2 や Qwen, SmolLM は因果言語モデル（自己回帰ともいう）と呼ばれる。先行したトークンが与えられた時に、シーケンスの次のトークンを予測するように訓練されていることから。

この節ではモデルがどのように予測しているのかを見てみる。

transformer ライブラリには多くのモデルとそれに対応するトークナイザーを備えているが、個別に指定する必要はない。なぜならAutoTokenizer や AutoModelForCausalLM によって処理に適したものが選定されるから。

In [27]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")

In [28]:
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model(input_ids)
outputs.logits.shape # 各入力トークンを出力する

# torch.Size([1, 7, 151936])
# バッチ数、シーケンス長（トークン数）、語彙数（トークンごとの判定に用いられる語彙数（１５万））

torch.Size([1, 10, 151936])

ロジットとは？ Gemini より

言語モデルの文脈における「ロジット」とは、モデルが次のトークンを予測する際に出力する生の値のことです。これらの値はまだ確率に変換されていません。

具体的には、モデルが「It was a dark and stormy」という入力を受け取った後、次に続く可能性のあるすべての単語（語彙内のすべてのトークン）に対して、どれくらいその単語が適切かを示す数値を出力します。この数値がロジットです。

例えば、outputs.logits.shapeがtorch.Size([1, 7, 151936])という出力だった場合、これは以下を意味します。

1: バッチサイズ（入力が1つであるため）。
7: シーケンス長（入力トークンの数、「It was a dark and stormy」が7トークンに分解されたため）。
151936: 語彙サイズ（モデルが知っているすべての単語の数）。
つまり、各入力トークン（例: 'stormy'の前の'storm'）に対して、次に続く可能性のある151936個のすべての単語それぞれにロジット値が割り当てられます。これらのロジット値は、正負の無限大までの範囲をとることができ、高い値ほどそのトークンが次に続く可能性が高いことを示唆します。

最終的に、これらのロジットはソフトマックス関数（または同様の関数）を通過することで、0から1の間の確率値に変換され、その確率に基づいて最も可能性の高い次のトークンが選択されます。

In [29]:
final_logits = model(input_ids).logits[0, -1] # 最後のトークンに対応するロジット
final_logits.argmax() # 最大値のインデックス

# output -> tensor(3729)

tensor(12884)

In [30]:
tokenizer.decode(final_logits.argmax())

# output -> ' night'
# 次に続く単語が night である確率が最も高いということがわかる（ロジットの最大値がこれってこと）

' sky'

topk() メソッドを使って、他にどのようなトークンが候補に上がっていたかを見てみる。

In [31]:
import torch
top10_logits = torch.topk(final_logits, 10)
for index in top10_logits.indices:
  print(tokenizer.decode(index))

#  night
#  evening
#  day
#  morning
#  winter
#  afternoon
#  Saturday
#  Sunday
#  Friday
#  October

 sky
 wind
 storm
 rain
 moon
 weather
 sun
 stars
 only
 air


候補となっていたトークンがどれくらいの確率だったかをみるために、ロジットを確率に変換してみる。そのために softmax() で正規化する。

In [32]:
top10 = torch.topk(final_logits.softmax(dim=0), 10)
for value, index in zip(top10.values, top10.indices):
  print(f"{tokenizer.decode(index):<10} {value.item():.2%}")

 sky       9.40%
 wind      8.00%
 storm     4.90%
 rain      3.83%
 moon      2.28%
 weather   2.23%
 sun       2.23%
 stars     1.94%
 only      1.81%
 air       1.80%


ここまでの処理を input の文章を変えてみて色々試してみるとロジットの確率も結構変化することがわかった。